# PBI 07 — Nithish's fraud-model experiments

**Goal.** Give Trust & Safety evidence about which orders to send for human review. This is an experiment, not permission to automatically reject orders or customers.

**Historical results preserved:** the 15 November 2026 table and MLflow runs below were recorded before the shared preprocessing refactor. They are retained as historical evidence and are not presented as results from the shared eight-feature pipeline.

The current cells call the reusable PBI 03 eight-feature builder and preprocessor. Run All uses only development-eligible training and validation rows. It does not read or evaluate sealed test records.

## Historical PBI 07 results (pre-shared-preprocessor run)

These are the prior validation results, retained as historical evidence. They were generated before the shared preprocessing refactor and are not recomputed by the cells below. The listed run IDs are absent from the local MLflow database available in this checkout, so their original artifacts could not be reloaded here; the metrics and IDs remain recorded in `reports/PBI07_model_comparison.md`.

| Historical version | Threshold | F1 | Precision | Recall | AP | ROC-AUC | FP | FN | Reported run ID |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| Logistic fixed threshold | 0.5000 | 0.4867 | 0.3535 | 0.7806 | 0.5079 | 0.8416 | 28,177 | 4,331 | `72e67b546e8546faa76a3f60479ac739` |
| Same logistic, calibrated threshold | 0.6425 | 0.5194 | 0.4797 | 0.5662 | 0.5079 | 0.8416 | 12,122 | 8,564 | `007bb93e5b34487a9e35aef4ba8130de` |
| Histogram gradient boosting | 0.6940 | 0.5390 | 0.5032 | 0.5803 | 0.5259 | 0.8499 | 11,311 | 8,285 | `7598ab24dbe14dfab74eeece9222efdf` |


## 1. Setup

All training logic lives in `src/train_nithish.py`. This notebook calls that routine and explains its outputs, so the notebook and the PBI 10 pipeline use the same implementation.

Set `MLFLOW_TRACKING_URI` to the shared service before running when you need shared evidence. Without it, the routine deliberately warns and logs locally; a local run does not satisfy the shared-MLflow acceptance criterion. Save each shared run link and verify that its model artifact can be loaded.

In [ ]:
import os, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://mlflow-pbi12.eba-9bnrxizu.us-east-1.elasticbeanstalk.com")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_access import load_development_tables
from src.data_cleaning import clean_customers, clean_orders, clean_labels
from src.train_nithish import FEATURES, VERSIONS, prepare_data, train_nithish_models

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
print("MLflow target:", tracking_uri if tracking_uri else "LOCAL ONLY — set MLFLOW_TRACKING_URI for shared evidence")

## 2. Data and split

The development-only loader excludes sealed-period order records before the PBI 02 cleaners, feature joins or transformations. The shared PBI 03 helper then preserves the established chronological training/validation dates. Test features and labels are never loaded into this notebook.

For a consistent team threshold protocol, the newest 20% of whole outer-training dates calibrates thresholds; the model fits on the older training dates and that same fitted model scores calibration and validation. Validation is used only for final comparison.

In [ ]:
customers_raw, orders_raw, labels_raw = load_development_tables()
customers = clean_customers(customers_raw)[0]
orders = clean_orders(orders_raw)[0]
labels = clean_labels(labels_raw)[0]

x_train, y_train, x_val, y_val, train_dates, report = prepare_data(customers, orders, labels)

pd.DataFrame({
    "rows": [len(x_train), len(x_val)],
    "from": [report["train_first_date"], report["validation_first_date"]],
    "to": [report["train_last_date"], report["validation_last_date"]],
    "fraud rate": [f"{y_train.mean():.1%}", f"{y_val.mean():.1%}"],
}, index=["training", "validation"])

**What this shows**

- The outer row counts and dates can be compared with PBI 03 because they come from the shared split helper.
- Fraud is the minority class. A model that predicts every order as legitimate can appear accurate while catching no fraud, so accuracy is not the headline metric.
- Fraud prevalence may change over time. Compare training and validation prevalence, then treat validation as a later-period check rather than a promise of future performance.
- PBI 07 uses only part of outer training to fit each model because a separate chronological slice is reserved for threshold calibration. Record the actual counts shown above.

## 3. Shared eight-feature schema

PBI 03 supplies this exact ordered schema to PBI 06, PBI 07 and PBI 08: `order_value`, `num_items_ordered`, `order_weekday`, `order_month`, `country_code`, `collect_type`, `payment_method`, and `days_since_first_order`. Customer history is joined and normalized in `src/feature_engineering.py`; this notebook does not build a separate feature pipeline.

| Feature | Source | Use? | Reason |
| --- | --- | --- | --- |
| `order_value`, `num_items_ordered` | order | Yes | Known at checkout; imputed/scaled by shared training-fitted preprocessing |
| `order_weekday`, `order_month` | order date | Yes | Calendar values known at order time |
| `country_code`, `collect_type`, `payment_method` | order | Yes | Shared categorical encoding; payment casing is lowercased |
| `days_since_first_order` | order date and customer first-order timestamp | Cautiously | Missing/future first-order timestamps become missing; source timing must be confirmed |
| `refund_value`, IDs, undated customer totals | order/customer | No | Leakage risk, join-only identity, or uncertain timing |

**Point-in-time limitation:** the field is described as the first recorded order, not confirmed account registration. Its source must be available as of the scored order. The incoming order data do not contain `customer_id`; prediction-time customer identity must come from a separate trusted mapping, not a fraud label file.

In [ ]:
train = x_train.assign(is_fraud=y_train)
buckets = pd.cut(train["days_since_first_order"], [-1, 0, 7, 30, 90, 365, 100000],
                 labels=["same day", "1-7 days", "8-30 days", "31-90 days", "91-365 days", "over 1 year"])
by_age = train.groupby(buckets, observed=True)["is_fraud"].agg(fraud_rate="mean", orders="size")

ax = by_age["fraud_rate"].plot.bar(color="#c0392b", figsize=(7, 3.5), rot=0)
ax.axhline(y_train.mean(), ls="--", color="grey")
ax.text(5.45, y_train.mean() + 0.01, f"overall {y_train.mean():.1%}", ha="right", color="grey")
ax.set(title="Fraud rate by days since first recorded order (training data only)", xlabel="", ylabel="fraud rate")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.tight_layout(); plt.show()
by_age

**What this shows**

- The chart measures fraud rates by time since first recorded order in the outer training period only.
- Check both the rate and the order count in every bucket. A striking rate based on very few records is weak business evidence.
- This is a relationship in historical data, not proof that the date is account registration or that this feature causes fraud.

In [ ]:
# Leakage check on training rows: does refund information give away the answer?
raw = labels.reset_index(drop=True).merge(orders, on=["country_code", "order_id"], how="left").loc[x_train.index]
(raw["refund_value"] > 0).groupby(y_train).mean().rename({0: "legit orders", 1: "fraud orders"}).to_frame("share with a refund")

**What this shows**

68% of legit orders have a refund and **0% of fraud orders do**. The refund column is filled in after the case is decided, so it practically *is* the label. A model using it would look brilliant here and fail in production, where the refund has not happened yet. It stays out.

## 4. Train all three versions

One call trains, evaluates and logs all versions using the shared eight-feature preprocessing pipeline. It appends new MLflow runs under a distinct `shared-eight-feature` batch name; the historical PBI 07 runs are never overwritten.

Thresholds for v2 and v3 are selected by maximum fraud F1 on the latest 20% of whole outer-training dates. Models fit on the older dates; the same fitted model scores calibration and validation. v1 shares the fitted logistic model with v2 and uses the fixed 0.50 threshold. Validation labels never select thresholds.

In [ ]:
results = train_nithish_models(customers, orders, labels, batch_name="shared-eight-feature")
by_version = {row["version"]: row for row in results}

summary = pd.DataFrame([{
    "version": row["version"], "cut-off": row["threshold"],
    "F1": row["metrics"]["f1"], "precision": row["metrics"]["precision"], "recall": row["metrics"]["recall"],
    "AP": row["metrics"]["average_precision"], "ROC-AUC": row["metrics"]["roc_auc"],
    "false alarms": row["metrics"]["fp"], "missed fraud": row["metrics"]["fn"],
    "orders reviewed": f"{row['metrics']['review_rate']:.1%}", "fit seconds": round(row["training_seconds"], 1),
    "MLflow run": row["run_id"],
} for row in results]).set_index("version")
summary

**How to read the metrics**

- **Precision**: among orders sent to review, how many are fraud. Low precision consumes analyst time and burdens legitimate customers.
- **Recall**: among all fraud orders in validation, how many the model flags. Low recall misses losses.
- **F1**: balances precision and recall at one threshold; it does not represent business costs.
- **Average precision (AP)** and **ROC-AUC**: describe ranking quality across thresholds. They help distinguish a model change from a threshold-only policy change.
- Review rate, false alarms and missed fraud show the operational volume behind each score.

No single metric picks the production threshold. The business must provide review capacity and the relative costs of missed fraud and unnecessary review.

## 5. Version 1 — logistic regression baseline

**Why start here.** Logistic regression is the simplest sensible fraud model. Each feature pushes the risk up or down by a fixed amount, like a points scorecard. An analyst can read it, and it trains in seconds.

Skewed amounts are log-transformed and scaled. Categories are one-hot encoded. Missing values are filled (median, or a "missing" category). Fraud rows are up-weighted so the model does not ignore the rare class. All of this is fitted on training data only.

In [ ]:
v1 = by_version["v1_logistic_default"]
names = v1["model"][0].get_feature_names_out()
weights = pd.Series(v1["model"][-1].coef_[0], index=names).sort_values()
top = pd.concat([weights.head(7), weights.tail(7)])

ax = top.plot.barh(color=["#2e86c1" if w < 0 else "#c0392b" for w in top], figsize=(7, 4.5))
ax.set(title="v1 scorecard: what lowers (blue) and raises (red) fraud risk", xlabel="weight")
plt.tight_layout(); plt.show()

**Interpret the regenerated chart only after the corrected notebook has run.** Check whether the direction of each coefficient is plausible, whether the first-order-history feature behaves as expected, and whether country effects warrant a fairness or policy review. A coefficient is an association within this model; it is not evidence of causation.

In [ ]:
m = v1["metrics"]
print(f"Orders flagged for review : {m['tp'] + m['fp']:,} of {len(y_val):,}  ({m['review_rate']:.0%} of all orders)")
print(f"Flags that are real fraud : {m['tp']:,}  ({m['precision']:.0%})")
print(f"Good customers flagged    : {m['fp']:,}  ({1 - m['precision']:.0%} of flags)")

check = pd.DataFrame({"score": v1["scores"], "fraud": y_val.to_numpy()})
check["score band"] = pd.cut(check["score"], [0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0])
check.groupby("score band", observed=True)["fraud"].agg(actual_fraud_rate="mean", orders="size")

### Diagnosis of v1

After rerunning, use the regenerated threshold-specific validation counts and score-band fraud rates to decide whether the fixed 0.50 operating point sends too many legitimate orders to review. Do not reuse the pre-correction counts: the model now uses the same training fit as the threshold-calibrated v2.

## 6. Version 2 — same fitted model, different review threshold

v2 is **not a separately trained model**. It shares v1's fitted logistic-regression pipeline and changes only the decision threshold, selected on the reserved calibration period. Its validation results isolate the effect of that operating-policy change.

In [ ]:
v2 = by_version["v2_logistic_tuned_cutoff"]
compare = pd.DataFrame({name: by_version[key]["metrics"] for name, key in
                        [("v1 (cut-off 0.50)", "v1_logistic_default"),
                         (f"v2 (cut-off {v2['threshold']:.2f})", "v2_logistic_tuned_cutoff")]}).T
compare[["f1", "precision", "recall", "average_precision", "fp", "fn", "review_rate"]]

**Measured outcome of the threshold change**

Read the regenerated v1/v2 validation table. The ranking metrics should be identical because the fitted model and validation scores are identical; precision, recall, F1, false alarms, missed fraud and review volume can change with the threshold. State the measured trade-off from the rerun and do not present maximum F1 as the approved business threshold.

### Diagnosis of v2

Use the regenerated per-country and payment-method evidence to assess where the logistic model may miss interactions. Include each group's support before interpreting differences. Do not assume that country-specific differences are fixed or caused by the model alone.

In [ ]:
top_methods = train["payment_method"].value_counts().head(6).index
grid = (train[train["payment_method"].isin(top_methods)]
        .pivot_table(index="payment_method", columns="country_code", values="is_fraud", aggfunc="mean"))
plt.figure(figsize=(7, 3.5))
sns.heatmap(grid, annot=True, fmt=".0%", cmap="Reds", cbar=False)
plt.title("Fraud rate by payment method and country (training data)"); plt.xlabel(""); plt.ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
def by_country(row, minimum_support=100):
    """Validation outcomes with support; small groups are not interpreted."""
    frame = pd.DataFrame({"country": x_val["country_code"].to_numpy(), "fraud": y_val.to_numpy(),
                          "flag": row["scores"] >= row["threshold"]})
    def summarize(group):
        legitimate = group["fraud"] == 0
        flagged = group["flag"]
        return pd.Series({
            "orders": len(group), "fraud rate": group["fraud"].mean(),
            "orders flagged": group["flag"].mean(),
            "precision": group.loc[flagged, "fraud"].mean() if flagged.any() else np.nan,
            "recall": group.loc[group["fraud"] == 1, "flag"].mean()
                      if (group["fraud"] == 1).any() else np.nan,
            "false-positive rate": group.loc[legitimate, "flag"].mean()
                                   if legitimate.any() else np.nan,
        })
    summary = frame.groupby("country", sort=True).apply(summarize)
    summary.loc[summary["orders"] < minimum_support,
                ["fraud rate", "orders flagged", "precision", "recall", "false-positive rate"]] = np.nan
    return summary

by_country(v2)

**Diagnosis for v3**

Use the regenerated evidence to decide whether the logistic model's additive structure leaves meaningful patterns unexplained. Change one model choice at a time, then compare v2 and v3 on the same outer validation period and metric definitions. A result from this experiment does not justify automated action against a country or customer group.

## 7. Version 3 — histogram gradient boosting

Histogram gradient boosting can model nonlinearities and feature combinations. It uses the shared one-hot encoded eight-feature inputs and the same outer date split, inner fit/calibration split and validation period as v1/v2. The previous native categorical handling and monotonic constraint were removed because the shared one-hot transformed layout does not preserve a safe mapping from original feature constraints to encoded columns. This changes the model inductive bias; compare this as a new model version, not a like-for-like repeat. The effect on validation performance is reported in the separately labeled shared-eight-feature results.

The model is deliberately bounded in size for a fast retraining routine. Runtime and quality claims must come from the corrected rerun in the target environment, not from the old notebook output.

In [ ]:
v3 = by_version["v3_boosting_tuned_cutoff"]
compare = pd.DataFrame({name: by_version[key]["metrics"] for name, key in
                        [("v2 logistic", "v2_logistic_tuned_cutoff"), ("v3 boosting", "v3_boosting_tuned_cutoff")]}).T
compare[["f1", "precision", "recall", "average_precision", "roc_auc", "fp", "fn", "review_rate"]]

**Measured outcome of the model change**

After rerunning, compare v2 and v3 using AP/ROC-AUC for ranking and the validation counts/review rate for their selected thresholds. Report the measured values, not an expected improvement. If the results are mixed, describe the trade-off rather than declaring a winner.

In [ ]:
country = pd.DataFrame({"v2 logistic": by_country(v2)["precision"], "v3 boosting": by_country(v3)["precision"]})
ax = country.plot.bar(rot=0, color=["#2e86c1", "#c0392b"], figsize=(7, 3.3))
ax.set(title="Precision inside each country (validation)", xlabel="", ylabel="flags that are real fraud")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.tight_layout(); plt.show()

### What drives v3?

Shuffle one feature at a time and see how much the ranking quality (AP) drops. A big drop means the model relies on that feature.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

sample = x_val.sample(30000, random_state=42)
importance = permutation_importance(v3["model"], sample, y_val.loc[sample.index], scoring="average_precision",
                                    n_repeats=3, random_state=42, n_jobs=2)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
pd.Series(importance.importances_mean, index=FEATURES).sort_values().plot.barh(ax=axes[0], color="#34495e")
axes[0].set(title="What v3 relies on", xlabel="drop in AP when the feature is shuffled")
PartialDependenceDisplay.from_estimator(v3["model"], sample, ["days_since_first_order"], ax=axes[1])
axes[1].set_title("Effect of days since first recorded order on the v3 score"); axes[1].set_xlim(0, 1500)
plt.tight_layout(); plt.show()

**Interpret the regenerated diagnostics.** Feature importance is a model-specific sensitivity measure, not a causal explanation. Check whether the top inputs make business sense and whether the monotonic effect is consistent with verified data definitions. Country importance calls for review of subgroup performance and policy; it does not justify geography-based automatic declines.

## 8. All versions side by side

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, label, colour in [("v1_logistic_default", "v1/v2 logistic", "#2e86c1"),
                           ("v3_boosting_tuned_cutoff", "v3 boosting", "#c0392b")]:
    precision, recall, _ = precision_recall_curve(y_val, by_version[key]["scores"])
    axes[0].plot(recall, precision, color=colour, label=label)
for key, marker in [("v1_logistic_default", "o"), ("v2_logistic_tuned_cutoff", "s"), ("v3_boosting_tuned_cutoff", "*")]:
    m = by_version[key]["metrics"]
    axes[0].scatter(m["recall"], m["precision"], marker=marker, s=110, color="black", zorder=3, label=key.split("_")[0])
axes[0].axhline(y_val.mean(), ls=":", color="grey", label="flagging at random")
axes[0].set(title="Precision vs recall (validation)", xlabel="recall (fraud caught)", ylabel="precision (flags that are fraud)")
axes[0].legend()

bars = summary[["F1", "precision", "recall", "AP"]].rename(index=lambda v: v.split("_")[0])
bars.T.plot.bar(ax=axes[1], rot=0, color=["#aab7b8", "#2e86c1", "#c0392b"])
axes[1].set(title="Validation metrics by version", ylim=(0, 1))
plt.tight_layout(); plt.show()

### Relation to PBI 06

New PBI 06 and PBI 07 comparisons use the same eight features, train-fitted preprocessor, chronological outer dates and latest-20%-of-training threshold calibration. Historical seven-feature PBI 06 and pre-refactor PBI 07 results remain separately labeled and are not controlled comparisons.

## 9. What this means for the Trust & Safety analyst

In [ ]:
def per_thousand(row):
    m = row["metrics"]; total = m["tp"] + m["fp"] + m["tn"] + m["fn"]
    scale = 1000 / total
    return {"orders to review": round((m["tp"] + m["fp"]) * scale), "of which real fraud": round(m["tp"] * scale),
            "good customers bothered": round(m["fp"] * scale), "fraud missed": round(m["fn"] * scale)}

pd.DataFrame({row["version"].split("_")[0]: per_thousand(row) for row in results}).T

**Business interpretation**

After rerunning, translate validation counts to a per-1,000-order scenario and show both orders sent to review and fraud missed. This helps the Trust & Safety lead compare model behavior with actual staff capacity. Do not extrapolate validation counts as guaranteed future savings.

**The cut-off is a business decision.** The notebook can show several review-volume scenarios, but the Trust & Safety lead must choose a policy using review capacity, missed-fraud costs and customer impact.

In [ ]:
scores = v3["scores"]; truth = y_val.to_numpy()
rows = []
for share in [0.05, 0.10, 0.15, 0.20, 0.30]:
    cut = np.quantile(scores, 1 - share); flagged = scores >= cut
    rows.append({"review the riskiest": f"{share:.0%} of orders", "cut-off": round(float(cut), 3),
                 "precision": truth[flagged].mean(), "recall": truth[flagged].sum() / truth.sum()})
pd.DataFrame(rows).set_index("review the riskiest")

## 10. Limits and honest caveats

- Fraud prevalence can drift; a later validation period does not guarantee future performance.
- A new payment method or market pattern can differ from training. Monitor performance and refresh the model deliberately.
- `first_order_datetime` is first recorded order time, not confirmed account creation. Verify the data dictionary before calling this account age.
- Undated customer totals remain excluded until the group proves they are available at order time.
- Maximum-F1 threshold is a comparison rule, not a business-approved threshold.
- Country is a sensitive operational signal. Inspect subgroup support, false-positive rates and review burden; use this model only to prioritise human review unless separate governance approves further action.
- The test period stays untouched until PBI 09 selects a candidate and the team agrees on a final test procedure.
- Shared MLflow evidence and a teammate review are required before PBI 07 can be marked complete.
- PBI 07 HistGradientBoosting now receives the shared one-hot representation. Its previous native categorical and monotonic constraints were removed because the encoded feature layout does not safely preserve those constraints; this changes the model's inductive bias and its results need to be interpreted as a new model version.
- The order CSV has no `customer_id`. Serving the history feature requires an independently available order-to-customer mapping; the label table is not an acceptable inference-time source.

## 11. How to use this

**Retrain (PBI 10 pipeline)** — pass the retained outputs of the PBI 02 cleaners:

```python
from src.train_nithish import train_nithish_models
results = train_nithish_models(customers, orders, labels, batch_name="new-batch")
```

Install the dependencies from `requirements-pbi07.txt`. Set `MLFLOW_TRACKING_URI` (or pass `tracking_uri`) to the shared service before a shared run. With no shared URI, logging is local-only and emits a warning. Each result includes the version, threshold, metrics, fitted pipeline, split evidence, and—when logging is enabled—the run ID and model URI. v1/v2 are one fitted model under two threshold policies, not separate trained model candidates.

**Before submission:** run every cell from top to bottom on the authorised base data; inspect the corrected outputs; log all versions to the shared MLflow service; verify artifact retrieval; save run links; and have a teammate review the evidence. Do not submit the cleared notebook as completed model evidence until this is done.
For PBI 08, import `prepare_training_data`, `fit_preprocessor` and `INPUT_FEATURES` from `src.feature_engineering`. Train Ephraim's estimator on `x_train` after fitting the shared preprocessor on `x_train` only; transform `x_val` with that same artifact. Use `train_dates` only as split/calibration metadata, never as an estimator input. No Ephraim notebook is present in this local branch.